# Nighttime Lights and Economic Activity: Evidence from India

**Using VIIRS satellite data to measure economic transformation across Indian regions**

This notebook walks through the complete analysis pipeline:
1. Understanding VIIRS nighttime lights data
2. Setting up the environment and defining regions of interest
3. Downloading data from Google Earth Engine (optional -- pre-downloaded data is included)
4. Analyzing Srinagar, Leh, and Manali (Kashmir/Ladakh focus)
5. Analyzing Ayodhya (the hockey-stick pattern)
6. Validating against official J&K economic data
7. How to add your own region

---

> Inspired by [Shedding Light on the Russia-Ukraine War](https://www.xkdr.org/paper/shedding-light-on-the-russia-ukraine-war) (xKDR Forum)

## 1. Introduction: Nighttime Lights as an Economic Proxy

Satellite-measured nighttime lights are a well-established proxy for economic activity. The key insight is simple: places with more economic activity have more artificial light at night. Factories, shops, street lights, and homes all contribute to the luminance captured by satellites.

### Why nighttime lights?

- **High frequency:** Monthly composites available (vs annual GDP data)
- **Spatially granular:** ~500m resolution, can analyze individual cities or neighborhoods
- **Objective:** Not subject to reporting biases in official statistics
- **Universal:** Same methodology works anywhere on Earth

### The VIIRS DNB sensor

We use data from the **Visible Infrared Imaging Radiometer Suite (VIIRS)** Day/Night Band (DNB) sensor aboard the Suomi NPP satellite (launched 2011). Specifically, we use the **monthly cloud-free composites** produced by NOAA's Earth Observation Group.

Key properties:
- **`avg_rad`**: Average radiance (nanoWatts/cm2/sr) -- this is our primary measure
- **`cf_cvg`**: Cloud-free coverage count -- how many cloud-free observations went into the composite
- Temporal coverage: April 2012 to present
- Spatial resolution: ~500m at the equator

### What we measure

For each region and month, we extract:
- **Total radiance (sum)**: Sum of all lit pixels in the bounding box. This captures both brightness and spatial extent.
- **Mean radiance**: Average brightness per lit pixel
- **Cloud-free observations**: Data quality indicator (low during monsoon months)

## 2. Setup

### Imports and configuration

We import functions from `ntl_analyze.py` (the reusable analysis library included in this repo) and set up the environment.

In [ ]:
import os
import sys
import glob
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import rasterio
from rasterio.mask import mask
from shapely.geometry import box, mapping

warnings.filterwarnings('ignore')

# Import our analysis library
from ntl_analyze import (
    REGIONS, SEASONS, TOURISM_SEASONS,
    process_region, add_season_columns, extract_radiance,
    plot_all_charts, print_summary,
    add_event_markers, get_baseline_and_labels,
    RASTER_DIR, OUTPUT_DIR
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

print(f'Data directory: {RASTER_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Available regions: {list(REGIONS.keys())}')

### Region definitions

Each region is defined by a bounding box `[west, south, east, north]` in decimal degrees. The regions currently defined in `ntl_analyze.py` are:

In [ ]:
for name, info in REGIONS.items():
    bbox = info['bbox']
    events = info.get('events', [])
    event_str = f'  Events: {", ".join(e["label"] for e in events)}' if events else ''
    print(f'{name:20s} {info["label"]:25s}  bbox={bbox}{event_str}')

## 3. Data Download (Optional)

**You do NOT need to run this section.** Pre-downloaded GeoTIFF files are included in `data/raster/`. This section is provided for reference, and for downloading data for new regions.

To download data, you need:
1. A [Google Earth Engine](https://earthengine.google.com/) account
2. A GEE project ID (set in `ntl_analyze.py` as `PROJECT_ID`)
3. Authentication via `earthengine authenticate`

In [ ]:
# ============================================================
# OPTIONAL: Uncomment to download data from Google Earth Engine
# ============================================================

# import ee
# ee.Initialize(project='YOUR-GEE-PROJECT-ID')
#
# from ntl_analyze import download_region
#
# # Download Srinagar data for 2014-2026
# download_region('srinagar', [(2014, 2026)])
#
# # Download Ayodhya data
# download_region('ayodhya', [(2017, 2026)])

Let's verify what data files are available:

In [ ]:
raster_files = sorted(glob.glob(os.path.join(RASTER_DIR, 'viirs_*.tif')))
raster_files = [f for f in raster_files if not f.endswith('_cf.tif')]
print(f'Total radiance files: {len(raster_files)}')

# Count files by region tag
from collections import Counter
tags = []
for f in raster_files:
    name = os.path.basename(f).replace('.tif', '')
    parts = name.split('_')
    if len(parts) == 3:  # viirs_YYYY_MM (generic/combined)
        tags.append('combined')
    else:  # viirs_REGION_YYYY_MM
        tags.append('_'.join(parts[1:-2]))

tag_counts = Counter(tags)
for tag, count in sorted(tag_counts.items()):
    print(f'  {tag:20s}: {count} months')

## 4. Analysis: Srinagar, Leh, and Manali

These three regions form the core of our Kashmir/Ladakh analysis:

- **Srinagar** -- capital of J&K, major urban center. Key event: Article 370 revocation (Aug 2019)
- **Leh** -- capital of Ladakh UT. Massive infrastructure buildup post-2019
- **Manali** -- control region in Himachal Pradesh with similar geography but no policy shock

### 4.1 Process data for all three regions

In [ ]:
# Process each region -- this extracts radiance from the GeoTIFF files
regions_to_analyze = ['srinagar', 'leh', 'manali']
all_dfs = []

for rkey in regions_to_analyze:
    print(f'Processing {REGIONS[rkey]["label"]}...')
    rdf = process_region(rkey)
    if not rdf.empty:
        all_dfs.append(rdf)
        print(f'  {len(rdf)} monthly records, {rdf["year"].min()}-{rdf["year"].max()}')

df_slm = pd.concat(all_dfs, ignore_index=True)
df_slm = add_season_columns(df_slm)
print(f'\nTotal records: {len(df_slm)}')

### 4.2 Time series

The raw monthly time series with 3-month moving average. Yellow shading marks summer (Apr-Jun), blue marks winter (Dec-Feb). The red dashed line marks the revocation of Article 370 on August 5, 2019.

In [ ]:
for rkey in regions_to_analyze:
    rinfo = REGIONS[rkey]
    rdf = df_slm[df_slm['region_key'] == rkey].sort_values('date').copy()
    if rdf.empty:
        continue

    fig, ax = plt.subplots(figsize=(15, 5))

    # Raw + smoothed
    ax.plot(rdf['date'], rdf['sum'], color=rinfo['color'], linewidth=1, alpha=0.4)
    rdf['sum_ma'] = rdf['sum'].rolling(window=3, min_periods=1).mean()
    ax.plot(rdf['date'], rdf['sum_ma'], color=rinfo['color'], linewidth=2.5,
            label=f'{rinfo["label"]} (3-month avg)')

    # Seasonal shading
    for year in rdf['year'].unique():
        ax.axvspan(datetime(year, 4, 1), datetime(year, 6, 30), alpha=0.08, color='gold')
        if datetime(year, 12, 1) <= rdf['date'].max():
            end_yr = min(year + 1, rdf['year'].max() + 1)
            ax.axvspan(datetime(year, 12, 1), datetime(end_yr, 2, 28), alpha=0.08, color='lightblue')

    add_event_markers(ax, rkey)

    ax.set_ylabel('Total Radiance', fontsize=12)
    ax.set_title(f'{rinfo["label"]} -- Nighttime Lights Time Series', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    plt.tight_layout()
    plt.show()

### 4.3 Seasonal patterns

Comparing monthly radiance patterns between the baseline period (earliest available years) and the most recent period. The percentage labels show growth in each month.

In [ ]:
for rkey in regions_to_analyze:
    rinfo = REGIONS[rkey]
    rdf = df_slm[df_slm['region_key'] == rkey].sort_values('date').copy()
    if rdf.empty:
        continue

    baseline, post, bl_label, post_label = get_baseline_and_labels(rdf)
    if baseline.empty or post.empty:
        continue

    fig, ax = plt.subplots(figsize=(10, 6))
    month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

    bl_monthly = baseline.groupby('month')['sum'].mean()
    post_monthly = post.groupby('month')['sum'].mean()

    x = np.arange(12)
    width = 0.35
    bl_vals = [bl_monthly.get(m, 0) for m in range(1, 13)]
    post_vals = [post_monthly.get(m, 0) for m in range(1, 13)]

    ax.bar(x - width/2, bl_vals, width, label=bl_label, color='#90CAF9', edgecolor='#1565C0')
    ax.bar(x + width/2, post_vals, width, label=post_label, color='#FFB74D', edgecolor='#E65100')

    for i, (bv, pv) in enumerate(zip(bl_vals, post_vals)):
        if bv > 0:
            pct = ((pv - bv) / bv) * 100
            ax.text(x[i] + width/2, pv + max(post_vals) * 0.02,
                    f'{pct:+.0f}%', ha='center', va='bottom', fontsize=7,
                    fontweight='bold', color='#E65100')

    ax.set_xticks(x)
    ax.set_xticklabels(month_labels)
    ax.set_ylabel('Avg Total Radiance', fontsize=12)
    ax.set_title(f'{rinfo["label"]} -- Monthly Pattern', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

### 4.4 Yearly trends

Annual average total radiance for each region. This smooths out seasonal variation and shows the long-term trajectory.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

for rkey in regions_to_analyze:
    rinfo = REGIONS[rkey]
    rdf = df_slm[df_slm['region_key'] == rkey].sort_values('date')
    if rdf.empty:
        continue

    yearly = rdf.groupby('year')['sum'].mean()
    ax.plot(yearly.index, yearly.values, color=rinfo['color'],
            marker=rinfo['marker'], linewidth=2.5, markersize=8,
            label=rinfo['label'])

# Article 370 line
ax.axvline(x=2019.6, color='red', linestyle='--', linewidth=2, label='Article 370 revoked')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Average Total Radiance', fontsize=12)
ax.set_title('Yearly Average Nighttime Lights -- Srinagar, Leh, Manali', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.5 Key observations

**Srinagar (+91% from baseline):**
- Steady growth across the full period
- No visible disruption from Article 370 revocation or internet shutdowns
- Winter lights growing faster than summer -- suggests year-round economic activity increasing

**Leh (+225% from baseline, now declining):**
- Dramatic surge 2019-2023, driven by military/infrastructure buildup after Ladakh became a UT
- Recent decline (2024-2025) as construction activity normalizes
- Extreme seasonality (summer 2-3x winter) reflecting Ladakh's harsh winter climate

**Manali (control):**
- Strong seasonal tourism signal but more moderate long-term growth
- No policy shock to respond to -- useful baseline for comparison

## 5. Analysis: Ayodhya -- The Hockey Stick

Ayodhya presents one of the clearest cases of policy-driven transformation visible from space. The Ram Temple project (Bhoomi Pujan August 2020, inauguration January 2024) triggered massive construction and infrastructure investment in a previously small town.

### 5.1 Process Ayodhya data

In [ ]:
df_ayodhya = process_region('ayodhya')
df_ayodhya = add_season_columns(df_ayodhya)
print(f'Ayodhya: {len(df_ayodhya)} records, {df_ayodhya["year"].min()}-{df_ayodhya["year"].max()}')

### 5.2 Time series

In [ ]:
rinfo = REGIONS['ayodhya']
rdf = df_ayodhya.sort_values('date').copy()

fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(rdf['date'], rdf['sum'], color=rinfo['color'], linewidth=1, alpha=0.4)
rdf['sum_ma'] = rdf['sum'].rolling(window=3, min_periods=1).mean()
ax.plot(rdf['date'], rdf['sum_ma'], color=rinfo['color'], linewidth=2.5,
        label='Ayodhya (3-month avg)')

# Event markers
add_event_markers(ax, 'ayodhya')

ax.set_ylabel('Total Radiance', fontsize=12)
ax.set_title('Ayodhya -- Nighttime Lights Time Series', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout()
plt.show()

### 5.3 Yearly trend -- the hockey stick

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

yearly = rdf.groupby('year')['sum'].mean()
ax.plot(yearly.index, yearly.values, color=rinfo['color'],
        marker='*', linewidth=2.5, markersize=12, label='Ayodhya')

add_event_markers(ax, 'ayodhya', date_axis=False)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Average Total Radiance', fontsize=12)
ax.set_title('Ayodhya -- Yearly Average Nighttime Lights (Hockey Stick)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(sorted(rdf['year'].unique()))
plt.tight_layout()
plt.show()

### 5.4 Key observations

**The hockey stick is unmistakable:**
- Flat radiance from 2017-2020 (~200 units)
- Sharp inflection starting 2021 as construction ramps up
- Accelerating growth through 2024-2025 as airport, hotels, roads, and temple complex come online
- This is a textbook example of nighttime lights capturing real economic transformation in near-real-time

## 6. Validation Against Official J&K Economic Data

A critical question: do nighttime lights actually correlate with official economic indicators? We validate using J&K GSDP data, power consumption, and per capita income.

### 6.1 Load official economic data

In [ ]:
# Load official J&K economic data
econ_path = os.path.join(os.path.dirname(RASTER_DIR), 'jk_ladakh_economic_data.csv')
econ = pd.read_csv(econ_path)
print(f'Economic indicators: {econ["indicator"].unique()}')
print(f'Fiscal years: {sorted(econ["fiscal_year"].unique())}')

In [ ]:
# Compute fiscal year averages for Srinagar + Leh combined (proxy for J&K)
df_combined = pd.concat([
    df_slm[df_slm['region_key'] == 'srinagar'],
    df_slm[df_slm['region_key'] == 'leh']
], ignore_index=True)

df_combined['fiscal_year'] = df_combined.apply(
    lambda r: f"{r['year']}-{str(r['year']+1)[2:]}" if r['month'] >= 4
    else f"{r['year']-1}-{str(r['year'])[2:]}", axis=1)

ntl_fy = df_combined.groupby('fiscal_year')['sum'].mean().reset_index()
ntl_fy.columns = ['fiscal_year', 'ntl_radiance']

# Extract key economic series
gsdp_current = econ[econ['indicator'] == 'GSDP_current_prices'][['fiscal_year', 'value']].copy()
gsdp_current.columns = ['fiscal_year', 'gsdp_current']

gsdp_constant = econ[econ['indicator'] == 'GSDP_constant_2011_12'][['fiscal_year', 'value']].copy()
gsdp_constant.columns = ['fiscal_year', 'gsdp_constant']

per_capita = econ[econ['indicator'] == 'per_capita_GSDP_current'][['fiscal_year', 'value']].copy()
per_capita.columns = ['fiscal_year', 'per_capita']

power_cost = econ[econ['indicator'] == 'power_purchase_cost'][['fiscal_year', 'value']].copy()
power_cost.columns = ['fiscal_year', 'power_cost']

# Merge
merged = ntl_fy.merge(gsdp_current, on='fiscal_year', how='outer') \
               .merge(gsdp_constant, on='fiscal_year', how='outer') \
               .merge(per_capita, on='fiscal_year', how='outer') \
               .merge(power_cost, on='fiscal_year', how='outer')
merged = merged.sort_values('fiscal_year')
merged_valid = merged.dropna(subset=['ntl_radiance', 'gsdp_current'])

print('Merged data (NTL + Official):')
print(merged_valid[['fiscal_year', 'ntl_radiance', 'gsdp_current',
                     'gsdp_constant', 'per_capita']].to_string(index=False))

### 6.2 Correlation analysis

In [ ]:
print('Correlation coefficients:')
for col, label in [('gsdp_current', 'GSDP (current prices)'),
                    ('gsdp_constant', 'GSDP (constant 2011-12)'),
                    ('per_capita', 'Per capita GSDP'),
                    ('power_cost', 'Power purchase cost')]:
    valid = merged_valid.dropna(subset=[col])
    if len(valid) >= 3:
        corr = valid['ntl_radiance'].corr(valid[col])
        print(f'  NTL vs {label}: r = {corr:.3f} (n={len(valid)})')

### 6.3 Visualization: NTL vs GSDP

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Dual-axis time series
ax1 = axes[0]
valid = merged_valid.dropna(subset=['gsdp_current'])
x = range(len(valid))
ax1.bar(x, valid['gsdp_current'] / 1000, color='#90CAF9', edgecolor='#1565C0',
        alpha=0.7, label="GSDP (current, Rs '000 Cr)")
ax1b = ax1.twinx()
ax1b.plot(list(x), valid['ntl_radiance'].values / 1000, color='#E53935', marker='o',
          linewidth=2.5, label="NTL Radiance ('000)")
ax1.set_xticks(list(x))
ax1.set_xticklabels(valid['fiscal_year'], rotation=45, fontsize=8)
ax1.set_ylabel("GSDP (Rs '000 Crore)", color='#1565C0', fontsize=11)
ax1b.set_ylabel("NTL Radiance ('000)", color='#E53935', fontsize=11)
corr = valid['ntl_radiance'].corr(valid['gsdp_current'])
ax1.set_title(f'NTL vs GSDP (Current Prices)\nr = {corr:.3f}', fontsize=13, fontweight='bold')
ax1.legend(loc='upper left', fontsize=8)
ax1b.legend(loc='upper right', fontsize=8)
ax1.grid(True, alpha=0.2)

# Panel 2: Scatter with regression
ax2 = axes[1]
ax2.scatter(valid['gsdp_current'] / 1000, valid['ntl_radiance'] / 1000,
            color='#1565C0', s=100, zorder=5, edgecolors='black')
for _, row in valid.iterrows():
    ax2.annotate(row['fiscal_year'],
                 (row['gsdp_current']/1000, row['ntl_radiance']/1000),
                 textcoords='offset points', xytext=(5, 5), fontsize=7)
z = np.polyfit(valid['gsdp_current']/1000, valid['ntl_radiance']/1000, 1)
p = np.poly1d(z)
x_line = np.linspace(valid['gsdp_current'].min()/1000, valid['gsdp_current'].max()/1000, 100)
ax2.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Linear fit (r={corr:.3f})')
ax2.set_xlabel("GSDP Current Prices (Rs '000 Crore)", fontsize=11)
ax2.set_ylabel("NTL Radiance ('000)", fontsize=11)
ax2.set_title('NTL vs GSDP -- Scatter Plot', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('Nighttime Lights vs Official Economic Data -- J&K (Srinagar + Leh)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.4 Normalized growth comparison

Indexing all series to 100 at the first overlapping year to compare growth trajectories.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

norm_base = merged_valid.iloc[0]
fiscal_years = merged_valid['fiscal_year'].values
x = range(len(fiscal_years))

for col, label, color, marker in [
    ('ntl_radiance', 'Nighttime Lights (Srinagar+Leh)', '#E53935', 'o'),
    ('gsdp_current', 'GSDP (current prices)', '#1565C0', 's'),
    ('gsdp_constant', 'GSDP (constant 2011-12)', '#2E7D32', '^'),
    ('per_capita', 'Per capita GSDP', '#FF9800', 'D'),
    ('power_cost', 'Power purchase cost', '#9C27B0', 'v'),
]:
    vals = merged_valid[col].values
    base_val = vals[0]
    if pd.isna(base_val) or base_val == 0:
        continue
    normalized = [(v / base_val * 100) if pd.notna(v) else np.nan for v in vals]
    ax.plot(list(x), normalized, color=color, marker=marker, linewidth=2, markersize=7, label=label)

ax.set_xticks(list(x))
ax.set_xticklabels(fiscal_years, rotation=45, fontsize=8)
ax.axhline(y=100, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.set_ylabel('Index (first year = 100)', fontsize=12)
ax.set_title('Growth Comparison: Nighttime Lights vs Official Economic Indicators\nJ&K (Srinagar + Leh)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.5 Validation summary

The correlations are remarkably strong:

| Indicator | Correlation (r) |
|---|---|
| GSDP at current prices | ~0.97 |
| GSDP at constant prices | ~0.93 |
| Power purchase cost | ~0.95 |
| Per capita GSDP | ~0.96 |

This validates that nighttime lights are a reliable proxy for economic activity in this region. The slightly lower correlation with constant-price GSDP makes sense -- nighttime lights capture nominal activity (more lights = more spending), which aligns better with current-price measures.

**Caveat:** Pre-2019 GSDP includes Ladakh; post-2019 excludes it (J&K UT only). Our NTL data covers Srinagar + Leh combined throughout, so the structural break in GSDP makes exact comparison imperfect after bifurcation.

## 7. How to Add Your Own Region

Adding a new region is straightforward. Here is a template:

In [ ]:
# Step 1: Define the region
# Find your bounding box at https://boundingbox.klokantech.com/ (use CSV format)

my_region = {
    'bbox': [73.72, 18.42, 73.98, 18.62],  # [west, south, east, north]
    'label': 'Pune (City)',
    'color': '#9C27B0',
    'marker': 'D',
    # Optional: add event markers
    # 'events': [{'date': datetime(2020, 1, 1), 'label': 'Some Event', 'color': 'red'}],
}

# Step 2: Add to REGIONS dict (or edit ntl_analyze.py directly)
# REGIONS['pune'] = my_region

# Step 3: Download data (requires GEE auth)
# from ntl_analyze import download_region
# download_region('pune', [(2017, 2026)])

# Step 4: Process and analyze
# df_pune = process_region('pune')
# df_pune = add_season_columns(df_pune)

# Step 5: Generate charts
# plot_all_charts(df_pune, ['pune'])
# print_summary(df_pune, ['pune'])

print('See ntl_analyze.py REGIONS dict for all pre-defined regions.')
print('Regions already defined:', list(REGIONS.keys()))

### Tips for choosing a bounding box

- **Too small:** Not enough pixels for a stable signal. Aim for at least ~20km x 20km for reliable aggregates.
- **Too large:** Dilutes the signal with surrounding rural areas. Match the urban extent of your area of interest.
- **Include buffer:** The download function adds a 0.1-degree margin automatically.
- **Verify visually:** Check your bbox on Google Maps or OpenStreetMap to make sure it covers the right area.

### Interpreting results

- **Monsoon months (Jun-Sep)** have poor cloud-free coverage in India. Check the `cf_mean` column -- values below 3 indicate unreliable data.
- **Total radiance (sum)** is usually more informative than mean radiance, as it captures both brightness increases and spatial expansion of lit areas.
- **Year-over-year trends** are more robust than month-to-month comparisons due to seasonal effects.

---

*Generated for the Nighttime Lights and Economic Activity project. See the [README](README.md) for full details.*